# Dự báo giao hàng: từ đơn hàng đến số ngày
**Sau bài 07 · 4–5 giờ, làm cá nhân hoặc cặp.** Đây là thời gian dự án bổ sung, không cộng vào 150 phút của bài học.

**Sản phẩm:** Pipeline dự báo thời gian giao hàng, so sánh baseline, xuất dự đoán và hàm phục vụ một đơn mới.

Đọc → dự đoán → chạy mã mẫu → tự điền TODO → kiểm thử → giải thích. Các TODO cố ý dừng bằng `NotImplementedError`; đây không phải lỗi môi trường. Không thay kiểm thử bằng giá trị viết cứng. Giữ seed, Restart Kernel và chạy lại trước khi nộp.

Nguồn, điều kiện sử dụng và giới hạn dữ liệu: [bảng nguồn](../../course/DATA_SOURCES.md). Không tải mô hình pickle/joblib không rõ nguồn.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mlcourse').is_dir()), None)
assert ROOT is not None, 'Mở notebook trong bản tải đầy đủ của repository.'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from threadpoolctl import threadpool_limits
_threads = threadpool_limits(limits=2)
SEED = 42

OUT = ROOT/'outputs/projects/03_delivery_estimator'
OUT.mkdir(parents=True, exist_ok=True)


## 1. Xác định lúc dự đoán
Dự đoán số ngày từ lúc duyệt đến lúc giao, dùng thông tin có thể biết khi duyệt đơn. Ngày giao thực tế chỉ để tạo nhãn/kiểm tra cutoff, không được đưa vào X.
Tập này chỉ gồm đơn giao thành công có đủ điều kiện: kết quả **không đại diện** cho đơn hủy/chưa giao. Không cam kết thời gian giao chính xác cho khách.


In [ ]:
from mlcourse.olist import prepare_olist, FEATURES
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
import joblib
# Đọc ZIP gốc, kiểm checksum; có thể cần một phút.
d, audit = prepare_olist(ROOT/'course/datasets')
print(audit)
train=d[d.split.eq('train')]; val=d[d.split.eq('validation')]; test=d[d.split.eq('test')]
assert d.order_id.is_unique
assert train.order_approved_at.max()<val.order_approved_at.min()
assert val.order_approved_at.max()<test.order_approved_at.min()
assert train.order_delivered_customer_date.lt(pd.Timestamp('2017-11-01')).all()
assert val.order_delivered_customer_date.lt(pd.Timestamp('2018-01-01')).all()
display(d.groupby('split').delivery_days.agg(['count','median','mean']))


## 2. TODO: sai số có đơn vị
MAE là trung bình trị tuyệt đối sai số, đơn vị ngày. Với thật [2,5], dự đoán [3,3], MAE=1.5 ngày.


In [ ]:
def mae(y, pred):
    y,pred=np.asarray(y,float),np.asarray(pred,float)
    if y.shape!=pred.shape or y.size==0: raise ValueError('Sai hình dạng hoặc rỗng')
    # TODO: trung bình trị tuyệt đối của hiệu.
    raise NotImplementedError('Hoàn thiện MAE')


In [ ]:
assert np.isclose(mae([2,5],[3,3]),1.5)
assert mae([1,2],[1,2])==0
assert np.isclose(mae([1],[4]),3)


## 3. Mã mẫu: baseline trung vị
Trung vị train là baseline phù hợp cho MAE. Thiếu cân nặng toàn bộ được chuyển thành NaN, không đánh đồng với cân nặng đo bằng 0. Median imputer nằm trong pipeline, chỉ fit train.


In [ ]:
def features(frame):
    x=frame[FEATURES].copy()
    x.loc[frame.known_weights.eq(0),'weight_g']=np.nan
    return x
Xt,Xv,Xe=map(features,[train,val,test])
yt,yv,ye=train.delivery_days,val.delivery_days,test.delivery_days
baseline=DummyRegressor(strategy='median').fit(Xt,yt)
print('Validation baseline MAE:',mae(yv,baseline.predict(Xv)))


## 4. TODO: học cây có giới hạn
Trả pipeline đã fit Xt, yt: SimpleImputer(strategy="median") → DecisionTreeRegressor(max_depth=depth, min_samples_leaf=30, random_state=SEED). Không fit trên Xv/Xe.


In [ ]:
def fit_candidate(depth):
    # TODO: tạo pipeline và fit trên Xt, yt.
    raise NotImplementedError('Hoàn thiện pipeline')


Gợi ý: `make_pipeline(...).fit(Xt, yt)`. Độ sâu lớn có thể giảm lỗi train nhưng tăng lỗi validation.


In [ ]:
models={depth:fit_candidate(depth) for depth in [2,4,6]}
scores=pd.DataFrame([{'depth':k,'train_mae':mae(yt,m.predict(Xt)),'validation_mae':mae(yv,m.predict(Xv))} for k,m in models.items()])
best_depth=int(scores.sort_values(['validation_mae','depth']).iloc[0]['depth'])
model=models[best_depth]
assert np.isfinite(model.predict(Xv)).all()
assert model[-1].get_depth()<=best_depth
display(scores)
scores.plot(x='depth',y=['train_mae','validation_mae'],marker='o')
plt.ylabel('MAE (days)'); plt.savefig(OUT/'learning_tradeoff.png'); plt.show()


## 5. Đánh giá cuối và đóng gói
Khóa độ sâu từ validation; test là khoảng thời gian muộn hơn. Khách hàng có thể lặp qua các khoảng: đây là đánh giá theo thời gian, không phải khẳng định tổng quát cho khách hàng chưa thấy.


In [ ]:
pred=model.predict(Xe)
metrics={'depth':best_depth,'test_mae_days':mae(ye,pred),'baseline_mae_days':mae(ye,baseline.predict(Xe)),'test_count':len(ye),'seed':SEED}
print(metrics)
pd.DataFrame({'order_id':test.order_id,'actual_days':ye,'predicted_days':pred}).to_csv(OUT/'predictions.csv',index=False)
(OUT/'metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
joblib.dump({'model':model,'features':FEATURES},OUT/'delivery_model.joblib')
plt.scatter(ye,pred,s=3,alpha=.2); plt.xlabel('Actual days'); plt.ylabel('Predicted days')
plt.savefig(OUT/'actual_predicted.png'); plt.show()


## 6. Demo đầu vào một đơn
Nhận đặc trưng đã tính từ đơn hàng, không nhận ngày giao thực tế. Đây là hàm dự đoán cục bộ, chưa phải dịch vụ production.


In [ ]:
bundle=joblib.load(OUT/'delivery_model.joblib')
def predict_delivery(order_features):
    missing=set(bundle['features'])-set(order_features)
    if missing: raise ValueError(f'Thiếu đặc trưng: {missing}')
    return float(bundle['model'].predict(pd.DataFrame([order_features])[bundle['features']])[0])
sample=Xe.iloc[0].to_dict()
print('Dự đoán số ngày:',predict_delivery(sample))
assert np.isclose(predict_delivery(sample),pred[0])


## Hoàn thiện sản phẩm và tự kiểm
Nguồn Olist có điều kiện phi thương mại. Không dùng ngày giao, late hoặc delivery_days làm đặc trưng. Mở rộng: hồi quy tuyến tính + scaling, hoặc thêm đặc trưng biết tại lúc duyệt; so sánh validation trước khi đánh giá test mới.

Nộp notebook đã chạy, thư mục đầu ra của dự án và `REPORT.md` tự viết: vấn đề → dữ liệu → phương pháp → kết quả → giới hạn. Dẫn nguồn, ghi seed/cách chia và mô tả ít nhất một trường hợp thất bại. Không chỉ chụp hình điểm số.

**Câu hỏi phản tư:** Điều gì trong kết quả là bằng chứng, điều gì mới là giả thuyết? Nếu dữ liệu mới khác dữ liệu học, bạn cần kiểm tra gì?

**Nhiệm vụ mở:** thay đổi một lựa chọn có lý do, xác định tiêu chí trước khi chạy và so sánh trên validation (hoặc phân tích độ ổn định với bài phân cụm). Không liên tục xem test để chọn cách làm.
